# Port Intelligence System — Phase 2 Pipeline (Google Colab)

**Purpose:** Run the full Phase 2 feature engineering pipeline on Google Colab using 31 days of NOAA AIS data.

**Run order:** Execute cells top to bottom in order. Do not skip any cell.

**Expected total runtime:** 25–45 minutes depending on Colab RAM allocation.

---
### Cell Sequence
1. Mount Google Drive
2. Verify project structure
3. Install dependencies
4. Download NOAA AIS days 6–31
5. Run `vessel_features.py`
6. Run `port_features.py`
7. Run `external_features.py`
8. Run `build_master_dataset.py`
9. Verify output files
10. Download processed files to laptop

---
## Cell 1 — Mount Google Drive

This connects Colab to your Google Drive. A popup will ask you to sign in and grant access. Click through and allow it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print('✓ Google Drive mounted at /content/drive')

Mounted at /content/drive
✓ Google Drive mounted at /content/drive


In [ ]:
import os

os.listdir('/content/drive/MyDrive/')

---
## Cell 2 — Set Project Root and Verify Structure


In [ ]:
import sys
import os
from pathlib import Path

# ── UPDATE THIS IF YOUR FOLDER NAME IS DIFFERENT ──────────────────────────────
FOLDER_NAME = 'port-intelligence-colab'
# ──────────────────────────────────────────────────────────────────────────────

PROJECT_ROOT = f'/content/drive/MyDrive/{FOLDER_NAME}'

# Add project root to Python path so src/ imports work
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Set working directory to project root
os.chdir(PROJECT_ROOT)

print(f'Project root : {PROJECT_ROOT}')
print(f'Working dir  : {os.getcwd()}')
print()

# ── Verify critical files and folders exist ───────────────────────────────────
checks = {
    'config.yaml'                          : 'config.yaml',
    'src/config_loader.py'                 : 'config loader',
    'src/features/vessel_features.py'      : 'vessel_features.py',
    'src/features/port_features.py'        : 'port_features.py',
    'src/features/external_features.py'    : 'external_features.py',
    'src/pipeline/build_master_dataset.py' : 'build_master_dataset.py',
    'data/raw/ais'                         : 'AIS raw folder',
    'data/raw/weather'                     : 'weather raw folder',
    'data/processed'                       : 'processed folder (empty ok)',
}

all_ok = True
for rel_path, label in checks.items():
    full_path = Path(PROJECT_ROOT) / rel_path
    exists    = full_path.exists()
    icon      = '✓' if exists else '✗ MISSING'
    print(f'  {icon}  {label:<40} {rel_path}')
    if not exists:
        all_ok = False

print()
if all_ok:
    print('✓ All required files and folders found. Ready to proceed.')
else:
    print('✗ Some files are missing. Fix the above before continuing.')

# Show existing AIS files
ais_dir   = Path(PROJECT_ROOT) / 'data/raw/ais'
ais_files = sorted(ais_dir.glob('*.csv'))
print(f'\nExisting AIS files ({len(ais_files)}):')
for f in ais_files:
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:<30} {size_mb:.1f} MB')

---
## Cell 3 — Install Dependencies

Colab has pandas and numpy pre-installed. We only need to add pyarrow and pyyaml.

In [ ]:
import os

os.makedirs("data/raw/ais", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("✓ Missing folders recreated")

✓ Missing folders recreated


In [ ]:
print('Installing dependencies ...')
!pip install pyarrow==16.0.0 pyyaml==6.0.1 --quiet

# Verify imports work
import pandas as pd
import numpy as np
import pyarrow
import yaml

print(f'✓ pandas     {pd.__version__}')
print(f'✓ numpy      {np.__version__}')
print(f'✓ pyarrow    {pyarrow.__version__}')
print(f'✓ yaml       {yaml.__version__}')
print()

# Test config_loader import — this is the most common failure point
try:
    from src.config_loader import load_config
    config = load_config()
    print(f'✓ config_loader imported successfully')
    print(f'  Ports configured : {[p["name"] for p in config["ports"]]}')
    print(f'  Date range       : {config["data"]["start_date"]} → {config["data"]["end_date"]}')
    print(f'  Anchorage radius : {config["anchorage"]["radius_km"]} km')
    if config['anchorage']['radius_km'] != 25:
        print(f'  ⚠ WARNING: radius_km is {config["anchorage"]["radius_km"]} — expected 25. Update config.yaml.')
    else:
        print(f'  ✓ radius_km = 25 confirmed')
except Exception as e:
    print(f'✗ config_loader import failed: {e}')
    print('  Check that src/config_loader.py exists and PROJECT_ROOT is set correctly.')

Installing dependencies ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 725.0/725.0 kB 43.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires PyYAML<7.0.0,>=6.0.2, but you have pyyaml 6.0.1 which is incompatible.
✓ pandas     2.2.2
✓ numpy      2.0.2
✓ pyarrow    18.1.0
✓ yaml       6.0.1

✓ config_loader imported successfully
  Ports configured : ['Los Angeles', 'Rotterdam', 'Singapore', 'Shanghai', 'Hamburg']
  Date range       : 2023-01-01 → 2023-12-31
  Anchorage radius : 25 km
  ✓ radius_km = 25 confirmed


---
## Cell 4 — Download NOAA AIS Data

Downloads remaining January 2023 AIS files directly from NOAA into your Drive.

- Skips files you already have (days 1–5)
- Downloads zip, extracts CSV, deletes zip automatically
- If a file fails, it logs the error and continues — you can re-run this cell safely

In [ ]:
# Cell 4 — Download NOAA AIS Data (January + February + March 2023)
#
# Skips any files already downloaded.
# Only fetches new files — February and March.
# Expected time: 20-35 minutes for new files only

import urllib.request
import zipfile
import time
from pathlib import Path

ais_dir  = Path(PROJECT_ROOT) / 'data/raw/ais'
base_url = 'https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/'

# Define exactly which months and days to download
MONTHS = [
    (1,  31, 'January'),
    (2,  28, 'February'),   # 2023 is not a leap year
    (3,  31, 'March'),
]

results = []
print(f'NOAA AIS Download — January + February + March 2023')
print(f'Target directory: {ais_dir}')
print(f'{"─"*60}')

for month_num, days_in_month, month_name in MONTHS:
    print(f'\n  {month_name} (month {month_num:02d}):')

    for day in range(1, days_in_month + 1):
        filename = f'AIS_2023_{month_num:02d}_{day:02d}'
        csv_path = ais_dir / f'{filename}.csv'
        zip_path = ais_dir / f'{filename}.zip'

        # Skip if CSV already exists
        if csv_path.exists():
            size_mb = csv_path.stat().st_size / 1e6
            print(f'    – {filename}  already exists ({size_mb:.0f} MB) — skipping')
            results.append({'file': filename, 'status': 'skipped'})
            continue

        url = base_url + f'{filename}.zip'
        print(f'    ↓ {filename}  downloading ...', end=' ', flush=True)

        try:
            urllib.request.urlretrieve(url, zip_path)

            with zipfile.ZipFile(zip_path, 'r') as z:
                z.extractall(ais_dir)

            zip_path.unlink()

            size_mb = csv_path.stat().st_size / 1e6 if csv_path.exists() else 0
            print(f'✓  ({size_mb:.0f} MB)')
            results.append({'file': filename, 'status': 'downloaded', 'size_mb': size_mb})

        except Exception as e:
            if zip_path.exists():
                zip_path.unlink()
            print(f'✗  FAILED: {e}')
            results.append({'file': filename, 'status': 'failed', 'error': str(e)})

        time.sleep(0.5)

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'\n{"─"*60}')
downloaded = [r for r in results if r['status'] == 'downloaded']
skipped    = [r for r in results if r['status'] == 'skipped']
failed     = [r for r in results if r['status'] == 'failed']

print(f'Downloaded : {len(downloaded)} new files')
print(f'Skipped    : {len(skipped)} files (already existed)')
print(f'Failed     : {len(failed)} files')

if failed:
    print(f'\n  Failed files:')
    for r in failed:
        print(f'    ✗ {r["file"]} — {r.get("error", "unknown error")}')
    print(f'\n  Re-run this cell to retry failed files.')

# Final inventory
all_csvs   = sorted(ais_dir.glob('*.csv'))
total_size = sum(f.stat().st_size for f in all_csvs) / 1e6

print(f'\nTotal AIS files in folder : {len(all_csvs)}')
print(f'Total size on disk        : {total_size:.0f} MB')

# Breakdown by month
jan = [f for f in all_csvs if 'AIS_2023_01' in f.name]
feb = [f for f in all_csvs if 'AIS_2023_02' in f.name]
mar = [f for f in all_csvs if 'AIS_2023_03' in f.name]

print(f'\n  January  : {len(jan):>2} files')
print(f'  February : {len(feb):>2} files')
print(f'  March    : {len(mar):>2} files')

expected = 31 + 28 + 31   # 90 total
if len(all_csvs) >= expected:
    print(f'\n✓ All {expected} expected files present. Proceed to Cell 5.')
else:
    missing = expected - len(all_csvs)
    print(f'\n⚠ {missing} files still missing. Re-run this cell to retry.')

NOAA AIS Download — January + February + March 2023
Target directory: /content/drive/MyDrive/port-intelligence-colab/data/raw/ais
────────────────────────────────────────────────────────────

  January (month 01):
    – AIS_2023_01_01  already exists (877 MB) — skipping
    – AIS_2023_01_02  already exists (846 MB) — skipping
    – AIS_2023_01_03  already exists (836 MB) — skipping
    – AIS_2023_01_04  already exists (846 MB) — skipping
    – AIS_2023_01_05  already exists (770 MB) — skipping
    – AIS_2023_01_06  already exists (912 MB) — skipping
    – AIS_2023_01_07  already exists (913 MB) — skipping
    – AIS_2023_01_08  already exists (890 MB) — skipping
    – AIS_2023_01_09  already exists (895 MB) — skipping
    – AIS_2023_01_10  already exists (875 MB) — skipping
    – AIS_2023_01_11  already exists (839 MB) — skipping
    – AIS_2023_01_12  already exists (823 MB) — skipping
    – AIS_2023_01_13  already exists (825 MB) — skipping
    – AIS_2023_01_14  already exists (868 MB)

In [ ]:
# Debug version of Cell 4b

import subprocess

result = subprocess.run(
    ['python', 'src/ingestion/fetch_weather.py'],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print("RETURN CODE:", result.returncode)

print("\nSTDOUT:\n")
print(result.stdout)

print("\nSTDERR:\n")
print(result.stderr)

RETURN CODE: 0

STDOUT:

[Weather] Starting weather ingestion for 5 ports
[Weather] Date range: 2023-01-01 → 2023-12-31
[Weather] Output: /content/drive/MyDrive/port-intelligence-colab/data/raw/weather

  [Weather] Los Angeles — already exists, skipping.
  [Weather] Fetching Houston (2023-01-01 → 2023-12-31) ...
  [Weather] Houston — ✓ 8,760 rows saved → weather_houston.csv
  [Weather] Fetching Savannah (2023-01-01 → 2023-12-31) ...
  [Weather] Savannah — ✓ 8,760 rows saved → weather_savannah.csv
  [Weather] Fetching Seattle (2023-01-01 → 2023-12-31) ...
  [Weather] Seattle — ✓ 8,760 rows saved → weather_seattle.csv
  [Weather] Fetching New York (2023-01-01 → 2023-12-31) ...
  [Weather] New York — ✓ 8,760 rows saved → weather_new_york.csv

[Weather] ── Ingestion Summary ─────────────────────────────────
  –  Los Angeles          skipped
  ✓  Houston              success  (8,760 rows)
  ✓  Savannah             success  (8,760 rows)
  ✓  Seattle              success  (8,760 rows)
  ✓  Ne

In [ ]:
from pathlib import Path
import pandas as pd

weather_dir = Path(PROJECT_ROOT) / "data/raw/weather"

for f in sorted(weather_dir.glob("*.csv")):
    df = pd.read_csv(f)

    print(f"\n{f.name}")
    print(f"Rows      : {len(df):,}")

    # Try common datetime column names
    for col in ["date", "datetime", "time", "timestamp"]:
        if col in df.columns:
            print(f"Min date  : {df[col].min()}")
            print(f"Max date  : {df[col].max()}")
            break


weather_hamburg.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_houston.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_los_angeles.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_new_york.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_rotterdam.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_savannah.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_seattle.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_shanghai.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00

weather_singapore.csv
Rows      : 8,760
Min date  : 2023-01-01T00:00
Max date  : 2023-12-31T23:00


In [ ]:
# Cell 4b — Fetch weather for new US ports

import subprocess
result = subprocess.run(
    ['python', 'src/ingestion/fetch_weather.py'],
    cwd=PROJECT_ROOT,
    capture_output=False,
    text=True
)

if result.returncode == 0:
    from pathlib import Path
    weather_dir   = Path(PROJECT_ROOT) / 'data/raw/weather'
    weather_files = list(weather_dir.glob('*.csv'))
    print(f'\n✓ Weather files ready: {len(weather_files)}')
    for f in sorted(weather_files):
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name:<45} {size_mb:.1f} MB')
else:
    print(f'✗ fetch_weather.py failed with code {result.returncode}')


✓ Weather files ready: 9
  weather_hamburg.csv                           0.5 MB
  weather_houston.csv                           0.5 MB
  weather_los_angeles.csv                       0.5 MB
  weather_new_york.csv                          0.5 MB
  weather_rotterdam.csv                         0.5 MB
  weather_savannah.csv                          0.5 MB
  weather_seattle.csv                           0.5 MB
  weather_shanghai.csv                          0.5 MB
  weather_singapore.csv                         0.5 MB


---
## Cell 5 — Run `vessel_features.py`

**What it does:** Loads all AIS CSVs, engineers vessel behavioral features (speed drop, anchorage flags, vessel class, etc.), saves `vessel_features.parquet`.

**Expected time: 15–25 minutes** — this is the heaviest cell, processing millions of rows.

Do not interrupt it. Watch the progress logs.

In [ ]:
import json
from pathlib import Path

progress_file = Path(PROJECT_ROOT) / 'data/processed/vessel_features_progress.json'
chunks_dir    = Path(PROJECT_ROOT) / 'data/processed/vessel_features_chunks'

if progress_file.exists():
    progress_file.unlink()
    print('✓ Progress checkpoint reset')

if chunks_dir.exists():
    import shutil
    shutil.rmtree(chunks_dir)
    chunks_dir.mkdir()
    print('✓ Chunks directory cleared')

print('Ready for fresh Cell 5 run with updated port config.')

Ready for fresh Cell 5 run with updated port config.


In [ ]:
# Cell 5 — Vessel Features (Chunked + Checkpointed + Streaming Merge)
#
# Processes one AIS CSV at a time (~500 MB RAM peak per file).
# Saves progress after every file. If Colab disconnects, re-run
# this cell — it skips already-completed files automatically.
#
# Merge uses PyArrow streaming writer — never loads more than
# one chunk into memory at a time regardless of total dataset size.
#
# Progress tracked in : data/processed/vessel_features_progress.json
# Chunks saved to     : data/processed/vessel_features_chunks/
# Final output        : data/processed/vessel_features.parquet

import json
import gc
import os
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
ais_dir       = Path(PROJECT_ROOT) / 'data/raw/ais'
processed_dir = Path(PROJECT_ROOT) / 'data/processed'
chunks_dir    = processed_dir / 'vessel_features_chunks'
progress_file = processed_dir / 'vessel_features_progress.json'
final_output  = processed_dir / 'vessel_features.parquet'

chunks_dir.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────────────────
COLUMN_MAP = {
    'MMSI': 'mmsi', 'BaseDateTime': 'timestamp', 'LAT': 'lat',
    'LON': 'lon', 'SOG': 'sog', 'COG': 'cog', 'Heading': 'heading',
    'VesselName': 'vessel_name', 'IMO': 'imo', 'VesselType': 'vessel_type',
    'Status': 'nav_status', 'Length': 'length', 'Width': 'width',
    'Draft': 'draft', 'Cargo': 'cargo_type',
}
REQUIRED_COLUMNS   = ['mmsi', 'timestamp', 'lat', 'lon', 'sog']
ANCHORED_THRESHOLD = 0.5
SLOW_THRESHOLD     = 3.0
ROLLING_WINDOW     = 12
MIN_PINGS          = 3

VESSEL_TYPE_MAP = {
    70: 'cargo',  71: 'cargo',  72: 'cargo',  73: 'cargo',  74: 'cargo',
    75: 'cargo',  76: 'cargo',  77: 'cargo',  78: 'cargo',  79: 'cargo',
    80: 'tanker', 81: 'tanker', 82: 'tanker', 83: 'tanker', 84: 'tanker',
    85: 'tanker', 86: 'tanker', 87: 'tanker', 88: 'tanker', 89: 'tanker',
    60: 'passenger', 61: 'passenger', 62: 'passenger', 63: 'passenger',
    64: 'passenger', 65: 'passenger', 66: 'passenger', 67: 'passenger',
    30: 'fishing', 21: 'tug_service', 22: 'tug_service',
    40: 'high_speed', 41: 'high_speed', 42: 'high_speed',
    36: 'pleasure', 37: 'pleasure', 35: 'military', 55: 'law_enforcement',
}

# ── Load progress tracker ─────────────────────────────────────────────────────
if progress_file.exists():
    with open(progress_file, 'r') as f:
        progress = json.load(f)
    print(f'Resuming from checkpoint — {len(progress["completed"])} files already done.')
else:
    progress = {'completed': [], 'failed': []}
    print('Starting fresh — no checkpoint found.')

# ── Get all AIS files ─────────────────────────────────────────────────────────
all_files = sorted(ais_dir.glob('*.csv'))
remaining = [f for f in all_files if f.name not in progress['completed']]

print(f'\nTotal AIS files     : {len(all_files)}')
print(f'Already processed   : {len(progress["completed"])}')
print(f'Remaining this run  : {len(remaining)}')
print(f'{"─"*55}')

# ── Feature engineering function ─────────────────────────────────────────────
def process_one_file(csv_path: Path) -> pd.DataFrame | None:
    """Load one AIS CSV, engineer all vessel features, return DataFrame."""

    df = pd.read_csv(csv_path, low_memory=False,
                     usecols=lambda c: c in COLUMN_MAP)
    df.rename(columns=COLUMN_MAP, inplace=True)

    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        print(f'    ⚠ Missing columns {missing} — skipping file')
        return None

    df.dropna(subset=REQUIRED_COLUMNS, inplace=True)
    if len(df) == 0:
        return None

    # Timestamps
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df.dropna(subset=['timestamp'], inplace=True)

    # Time features
    df['hour_of_day']  = df['timestamp'].dt.hour
    df['weekday']      = df['timestamp'].dt.dayofweek
    df['weekend_flag'] = (df['weekday'] >= 5).astype(np.int8)

    # SOG
    df['sog'] = pd.to_numeric(df['sog'], errors='coerce').fillna(0).clip(0, 50)

    # Speed flags
    df['anchored_flag']    = (df['sog'] < ANCHORED_THRESHOLD).astype(np.int8)
    df['slow_moving_flag'] = (df['sog'] < SLOW_THRESHOLD).astype(np.int8)

    # Sort per vessel for rolling ops
    df.sort_values(['mmsi', 'timestamp'], inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Rolling median SOG → speed_drop_ratio
    rolling_med = (
        df.groupby('mmsi', sort=False)['sog']
        .transform(lambda x: x.rolling(ROLLING_WINDOW, min_periods=MIN_PINGS).median())
    )
    safe_med = rolling_med.replace(0, np.nan)
    df['speed_drop_ratio'] = (df['sog'] / safe_med).clip(upper=3.0).round(4)

    # Rolling std SOG → movement_intensity
    df['movement_intensity'] = (
        df.groupby('mmsi', sort=False)['sog']
        .transform(lambda x: x.rolling(ROLLING_WINDOW, min_periods=MIN_PINGS).std())
    ).round(4)

    # Idle duration proxy
    flag     = df['slow_moving_flag']
    group_id = df.groupby('mmsi', sort=False)['slow_moving_flag'].transform(
        lambda x: (x == 0).cumsum()
    )
    df['idle_duration_proxy'] = 0
    slow_mask = flag == 1
    df.loc[slow_mask, 'idle_duration_proxy'] = (
        df[slow_mask]
        .groupby(['mmsi', group_id[slow_mask]])
        .cumcount() + 1
    )
    df['idle_duration_proxy'] = df['idle_duration_proxy'].astype(np.int32)

    # Vessel class
    if 'vessel_type' in df.columns:
        vtype = pd.to_numeric(df['vessel_type'], errors='coerce').fillna(-1).astype(int)
        df['vessel_class'] = vtype.map(VESSEL_TYPE_MAP).fillna('other')
    else:
        df['vessel_class'] = 'unknown'

    return df


# ── Process each remaining file ───────────────────────────────────────────────
for i, csv_path in enumerate(remaining, 1):
    print(f'  [{i}/{len(remaining)}] {csv_path.name} ...', end=' ', flush=True)

    try:
        df = process_one_file(csv_path)

        if df is None or len(df) == 0:
            print('⚠ empty or skipped')
            progress['completed'].append(csv_path.name)
        else:
            chunk_path = chunks_dir / f'{csv_path.stem}.parquet'
            df.to_parquet(chunk_path, index=False, engine='pyarrow', compression='snappy')

            rows = len(df)
            size = chunk_path.stat().st_size / 1e6
            print(f'✓  {rows:,} rows  ({size:.1f} MB chunk)')
            progress['completed'].append(csv_path.name)

        with open(progress_file, 'w') as f:
            json.dump(progress, f)

    except Exception as e:
        print(f'✗  FAILED: {e}')
        progress['failed'].append(csv_path.name)
        with open(progress_file, 'w') as f:
            json.dump(progress, f)

    finally:
        try:
            del df
        except:
            pass
        gc.collect()

# ── Streaming merge — one chunk at a time, never accumulates in RAM ───────────
print(f'\n{"─"*55}')
chunk_files = sorted(chunks_dir.glob('*.parquet'))
print(f'Merging {len(chunk_files)} chunks → vessel_features.parquet')
print(f'Method: PyArrow streaming writer (constant RAM regardless of size)')
print(f'{"─"*55}')

if len(chunk_files) == 0:
    print('✗ No chunk files found. Check errors above.')
else:
    writer      = None   # ParquetWriter opened on first chunk, reused for all
    schema      = None   # inferred from first chunk, enforced on all subsequent
    total_rows  = 0
    total_size  = 0.0

    try:
        for j, chunk_path in enumerate(chunk_files, 1):
            # Read one chunk — this is the ONLY thing in RAM at this moment
            table = pq.read_table(chunk_path, memory_map=True)

            if writer is None:
                # First chunk — capture schema and open the writer
                # Schema is fixed here; all subsequent chunks must match
                schema = table.schema
                writer = pq.ParquetWriter(
                    final_output,
                    schema,
                    compression='snappy',
                )
                print(f'  Schema locked from first chunk ({len(schema)} columns)')

            else:
                # Cast subsequent chunks to match the first chunk's schema
                # This handles minor dtype differences between daily files
                # (e.g. one file has int32 where another has int64)
                try:
                    table = table.cast(schema)
                except pa.ArrowInvalid as cast_err:
                    print(f'  ⚠ Chunk {j} schema mismatch — attempting column-level cast: {cast_err}')
                    # Build a new table column by column, casting where possible
                    arrays = []
                    for field in schema:
                        if field.name in table.schema.names:
                            col = table.column(field.name)
                            try:
                                arrays.append(col.cast(field.type))
                            except Exception:
                                # Fill incompatible column with nulls rather than crash
                                arrays.append(pa.nulls(len(table), type=field.type))
                        else:
                            arrays.append(pa.nulls(len(table), type=field.type))
                    table = pa.table(
                        {field.name: arrays[k] for k, field in enumerate(schema)},
                        schema=schema,
                    )

            # Write this chunk's rows directly to disk — no accumulation
            writer.write_table(table)

            chunk_rows = len(table)
            total_rows += chunk_rows
            total_size  = final_output.stat().st_size / 1e6

            print(f'  [{j:02d}/{len(chunk_files)}] {chunk_path.name:<35} '
                  f'{chunk_rows:>9,} rows   file so far: {total_size:.1f} MB')

            # Release this chunk from memory immediately
            del table
            gc.collect()

    finally:
        # Always close the writer — even if something crashes mid-merge
        # An unclosed ParquetWriter produces a corrupt file
        if writer is not None:
            writer.close()
            print(f'\n  Writer closed cleanly.')

    # ── Final verification ────────────────────────────────────────────────────
    if final_output.exists():
        final_size = final_output.stat().st_size / 1e6

        # Read only metadata (no data loaded) to verify row count
        pq_meta    = pq.read_metadata(final_output)
        file_rows  = pq_meta.num_rows

        print(f'\n{"─"*55}')
        print(f'✓ vessel_features.parquet written successfully')
        print(f'  Total rows  : {file_rows:,}')
        print(f'  File size   : {final_size:.1f} MB')
        print(f'  Row groups  : {pq_meta.num_row_groups}')
        print(f'  Peak RAM    : ~500 MB  (one chunk at a time — never accumulated)')

        if progress['failed']:
            print(f'\n⚠ {len(progress["failed"])} file(s) failed during processing:')
            print(f'  {progress["failed"]}')
            print(f'  Re-run this cell to retry. Failed files will be processed')
            print(f'  and appended to existing chunks automatically.')
        else:
            print(f'\n✓ All {len(all_files)} AIS files processed and merged.')
            print(f'  Proceed to Cell 6.')
    else:
        print(f'✗ vessel_features.parquet was not created — check errors above.')

Resuming from checkpoint — 3 files already done.

Total AIS files     : 90
Already processed   : 3
Remaining this run  : 87
───────────────────────────────────────────────────────
  [1/87] AIS_2023_01_04.csv ... ✓  7,866,609 rows  (112.2 MB chunk)
  [2/87] AIS_2023_01_05.csv ... ✓  7,155,930 rows  (104.1 MB chunk)
  [3/87] AIS_2023_01_06.csv ... ✓  8,484,361 rows  (121.6 MB chunk)
  [4/87] AIS_2023_01_07.csv ... ✓  8,485,661 rows  (121.4 MB chunk)
  [5/87] AIS_2023_01_08.csv ... ✓  8,275,827 rows  (116.5 MB chunk)
  [6/87] AIS_2023_01_09.csv ... ✓  8,323,136 rows  (116.0 MB chunk)
  [7/87] AIS_2023_01_10.csv ... ✓  8,135,468 rows  (114.1 MB chunk)
  [8/87] AIS_2023_01_11.csv ... ✓  7,795,074 rows  (109.6 MB chunk)
  [9/87] AIS_2023_01_12.csv ... ✓  7,648,275 rows  (107.5 MB chunk)
  [10/87] AIS_2023_01_13.csv ... ✓  7,676,204 rows  (106.2 MB chunk)
  [11/87] AIS_2023_01_14.csv ... ✓  8,084,458 rows  (112.4 MB chunk)
  [12/87] AIS_2023_01_15.csv ... ✓  8,092,890 rows  (112.5 MB chunk)
 

In [ ]:
from pathlib import Path

chunks = Path(PROJECT_ROOT) / "data/processed/vessel_features_chunks"

if chunks.exists():
    files = list(chunks.glob("*.parquet"))
    print("Chunk files saved:", len(files))

progress = Path(PROJECT_ROOT) / "data/processed/vessel_features_progress.json"
print("Progress JSON exists:", progress.exists())

Chunk files saved: 90
Progress JSON exists: True


In [ ]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.0Gi        10Gi       2.0Mi       1.0Gi        11Gi
Swap:             0B          0B          0B


---
## Cell 6 — Run `port_features.py`

**What it does:** Spatially assigns vessel pings to ports using Haversine distance, aggregates to daily port-level congestion signals, computes rolling features and composite congestion score.

**Expected time: 3–8 minutes**

In [ ]:
"""
port_features.py
----------------
Phase 2 — Port-Level Congestion Feature Engineering

Reads vessel_features.parquet in streaming row-group batches using
PyArrow's native batch reader. Never loads the full dataset into RAM.
Each batch is spatially assigned to ports and aggregated to daily
port-level counts. Only the tiny daily aggregations accumulate in memory.


Output
------
    data/processed/port_features.parquet
        One row per port per day with congestion signals and rolling features.
"""

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config_loader import load_config

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
EARTH_RADIUS_KM       = 6371.0
ROLLING_DAYS          = 7
ANCHORED_THRESHOLD    = 0.5
SLOW_THRESHOLD        = 3.0
IDLE_DURATION_CAP     = 50

CONGESTION_WEIGHTS = {
    "slow_vessel_ratio": 0.45,
    "anchored_ratio":    0.35,
    "idle_duration_norm":0.20,
}

# Columns we actually need from vessel_features.parquet
# Requesting only these prevents PyArrow from loading unused columns
REQUIRED_COLUMNS = [
    "mmsi", "timestamp", "lat", "lon", "sog",
    "anchored_flag", "slow_moving_flag", "idle_duration_proxy",
]


# ---------------------------------------------------------------------------
# Haversine (vectorised NumPy)
# ---------------------------------------------------------------------------

def haversine_vectorised(
    lat1: np.ndarray,
    lon1: np.ndarray,
    lat2: float,
    lon2: float,
) -> np.ndarray:
    lat1_r = np.radians(lat1)
    lon1_r = np.radians(lon1)
    dlat   = np.radians(lat2) - lat1_r
    dlon   = np.radians(lon2) - lon1_r
    a = (np.sin(dlat / 2.0) ** 2
         + np.cos(lat1_r) * np.cos(np.radians(lat2)) * np.sin(dlon / 2.0) ** 2)
    return EARTH_RADIUS_KM * 2.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


# ---------------------------------------------------------------------------
# Assign one batch to ports and aggregate to daily counts
# ---------------------------------------------------------------------------

def aggregate_batch(
    batch_df: pd.DataFrame,
    ports: list,
    radius_km: float,
) -> pd.DataFrame | None:
    """
    Given one batch of vessel pings as a DataFrame:
      1. Compute distance from each ping to every port (Haversine).
      2. Keep only pings within radius_km of at least one port.
      3. Assign each ping to its nearest port.
      4. Group by (nearest_port, date) and compute daily sums/counts.

    Returns a small daily aggregation DataFrame, or None if no pings
    in this batch fall within any port zone.

    The returned DataFrame uses SUM aggregations intentionally —
    sums from multiple batches covering the same (port, date) can be
    combined correctly in the final reduce step. Averaging across
    batches would produce wrong results (you can't average averages
    without knowing the denominator).
    """
    if len(batch_df) == 0:
        return None

    lat_arr = batch_df["lat"].to_numpy(dtype=np.float64)
    lon_arr = batch_df["lon"].to_numpy(dtype=np.float64)

    # Distance matrix: shape (n_pings, n_ports)
    dist_matrix    = np.column_stack([
        haversine_vectorised(lat_arr, lon_arr, p["lat"], p["lon"])
        for p in ports
    ])
    nearest_idx    = np.argmin(dist_matrix, axis=1)
    nearest_dist   = dist_matrix[np.arange(len(batch_df)), nearest_idx]
    within_mask    = nearest_dist <= radius_km

    if within_mask.sum() == 0:
        return None

    batch_df = batch_df[within_mask].copy()
    batch_df["nearest_port"] = [ports[i]["name"] for i in nearest_idx[within_mask]]
    batch_df["date"]         = batch_df["timestamp"].dt.normalize()

    # Aggregate to daily port level using SUMS and COUNTS
    # We avoid mean() here because means cannot be merged across batches
    daily = (
        batch_df.groupby(["nearest_port", "date"])
        .agg(
            total_pings           = ("mmsi",               "count"),
            unique_mmsi_approx    = ("mmsi",               "nunique"),
            anchored_pings        = ("anchored_flag",       "sum"),
            slow_pings            = ("slow_moving_flag",    "sum"),
            sog_sum               = ("sog",                 "sum"),
            idle_duration_sum     = ("idle_duration_proxy", "sum"),
        )
        .reset_index()
    )

    return daily


# ---------------------------------------------------------------------------
# Stream through vessel_features.parquet batch by batch
# ---------------------------------------------------------------------------

def stream_aggregate_all_batches(
    parquet_path: Path,
    ports: list,
    radius_km: float,
) -> pd.DataFrame:
    """
    Open vessel_features.parquet and iterate over its row groups one at a
    time using PyArrow's ParquetFile batch reader.

    Each row group is converted to a pandas DataFrame, aggregated to
    daily port-level sums, then immediately discarded. Only the tiny
    daily aggregation DataFrames accumulate in memory.

    Peak RAM = size of one row group (~50-200 MB) + accumulated daily
               aggregations (negligible — at most ~155 rows × n_cols).

    Parameters
    ----------
    parquet_path : Path
        Path to vessel_features.parquet.
    ports : list of dict
        Port definitions from config.yaml.
    radius_km : float
        Anchorage zone radius in kilometres.

    Returns
    -------
    pd.DataFrame
        Combined daily aggregations across all batches, ready for
        final reduce step.
    """
    pf       = pq.ParquetFile(parquet_path)
    n_groups = pf.metadata.num_row_groups
    total_rows = pf.metadata.num_rows

    log.info(f"Streaming vessel_features.parquet")
    log.info(f"  Row groups : {n_groups}")
    log.info(f"  Total rows : {total_rows:,}")
    log.info(f"  Reading columns: {REQUIRED_COLUMNS}")

    # Only read the columns we need — saves significant I/O and RAM
    # Check which required columns actually exist in this parquet
    file_columns    = [s.name for s in pf.schema_arrow]
    columns_to_read = [c for c in REQUIRED_COLUMNS if c in file_columns]
    missing_cols    = [c for c in REQUIRED_COLUMNS if c not in file_columns]

    if missing_cols:
        log.warning(f"  Columns not found in parquet (will be skipped): {missing_cols}")

    if "lat" not in columns_to_read or "lon" not in columns_to_read:
        raise ValueError(
            "vessel_features.parquet is missing 'lat' or 'lon' columns.\n"
            "Cannot perform spatial assignment without coordinates.\n"
            "Re-run vessel_features.py to regenerate the parquet."
        )

    batch_results = []
    total_matched = 0

    for group_idx in range(n_groups):
        # Read one row group — this is the only large object in RAM
        table    = pf.read_row_group(group_idx, columns=columns_to_read)
        batch_df = table.to_pandas()

        # Free the PyArrow table immediately — we only need the pandas df
        del table

        # Parse timestamp if it came in as string or object
        if not pd.api.types.is_datetime64_any_dtype(batch_df["timestamp"]):
            batch_df["timestamp"] = pd.to_datetime(
                batch_df["timestamp"], errors="coerce"
            )
        batch_df.dropna(subset=["timestamp", "lat", "lon"], inplace=True)

        # Coerce numeric columns
        for col in ["sog", "anchored_flag", "slow_moving_flag", "idle_duration_proxy"]:
            if col in batch_df.columns:
                batch_df[col] = pd.to_numeric(batch_df[col], errors="coerce").fillna(0)

        # Aggregate this batch
        daily_batch = aggregate_batch(batch_df, ports, radius_km)

        matched_in_batch = len(batch_df) if daily_batch is not None else 0
        total_matched   += matched_in_batch

        if daily_batch is not None:
            batch_results.append(daily_batch)

        # Log progress every 5 row groups
        if (group_idx + 1) % 5 == 0 or group_idx == n_groups - 1:
            pct = (group_idx + 1) / n_groups * 100
            log.info(
                f"  Row group {group_idx+1:>3}/{n_groups}  ({pct:.0f}%)  "
                f"batches with port matches: {len(batch_results)}"
            )

        # Release batch DataFrame immediately
        del batch_df
        del daily_batch

    log.info(f"Streaming complete.")

    if not batch_results:
        raise RuntimeError(
            "No vessel pings fell within the anchorage radius of any port.\n"
            "Possible causes:\n"
            "  1. config.yaml radius_km may be too small — try 50 instead of 25\n"
            "  2. AIS data geographic coverage may not overlap with configured ports\n"
            "  3. lat/lon values may be null or corrupted in vessel_features.parquet\n"
            "Check the AIS data covers the Port of Los Angeles region."
        )

    # Combine all batch aggregations — this is just n_groups × ~155 tiny rows
    log.info(f"Combining {len(batch_results)} batch aggregation results ...")
    combined = pd.concat(batch_results, ignore_index=True)
    del batch_results

    return combined


# ---------------------------------------------------------------------------
# Reduce: combine batch sums into correct daily totals
# ---------------------------------------------------------------------------

def reduce_to_daily(combined: pd.DataFrame) -> pd.DataFrame:
    """
    Multiple batches may cover the same (nearest_port, date) combination
    because one day's pings may span multiple row groups. This step sums
    all batch contributions for each (port, date) pair into correct totals.

    Then derives ratio features from the sums:
        slow_vessel_ratio  = slow_pings / total_pings
        anchored_ratio     = anchored_pings / total_pings
        avg_speed_in_zone  = sog_sum / total_pings
        avg_idle_duration  = idle_duration_sum / total_pings

    Parameters
    ----------
    combined : pd.DataFrame
        Raw batch aggregations with sum columns.

    Returns
    -------
    pd.DataFrame
        One row per (nearest_port, date) with correct daily totals.
    """
    log.info("Reducing batch sums to daily port totals ...")

    daily = (
        combined.groupby(["nearest_port", "date"])
        .agg(
            total_pings        = ("total_pings",        "sum"),
            vessels_in_zone    = ("unique_mmsi_approx", "sum"),  # approx — sum of nunique
            anchored_pings     = ("anchored_pings",     "sum"),
            slow_pings         = ("slow_pings",         "sum"),
            sog_sum            = ("sog_sum",            "sum"),
            idle_duration_sum  = ("idle_duration_sum",  "sum"),
        )
        .reset_index()
    )

    # Derive ratio features from correct daily sums
    safe_pings = daily["total_pings"].replace(0, np.nan)

    daily["slow_vessel_ratio"]     = (daily["slow_pings"]    / safe_pings).round(4)
    daily["anchored_ratio"]        = (daily["anchored_pings"] / safe_pings).round(4)
    daily["avg_speed_in_zone"]     = (daily["sog_sum"]        / safe_pings).round(3)
    daily["avg_idle_duration"]     = (daily["idle_duration_sum"] / safe_pings).round(2)

    # Rename pings to cleaner names
    daily.rename(columns={
        "anchored_pings": "anchored_vessel_count",
        "slow_pings":     "slow_vessel_count",
    }, inplace=True)

    # Drop intermediate sum columns no longer needed
    daily.drop(columns=["sog_sum", "idle_duration_sum"], inplace=True)

    daily.sort_values(["nearest_port", "date"], inplace=True)
    daily.reset_index(drop=True, inplace=True)

    log.info(f"  Daily rows : {len(daily):,}")
    log.info(f"  Ports      : {daily['nearest_port'].unique().tolist()}")
    log.info(f"  Date range : {daily['date'].min().date()} → {daily['date'].max().date()}")

    return daily


# ---------------------------------------------------------------------------
# Rolling features
# ---------------------------------------------------------------------------

def add_rolling_port_features(daily: pd.DataFrame) -> pd.DataFrame:
    """Add 7-day rolling averages per port using groupby + transform."""
    log.info(f"Adding {ROLLING_DAYS}-day rolling features ...")

    def rolling_mean(s: pd.Series) -> pd.Series:
        return s.rolling(window=ROLLING_DAYS, min_periods=3).mean()

    rolling_map = {
        "vessels_7d_rolling":    "vessels_in_zone",
        "anchored_7d_rolling":   "anchored_vessel_count",
        "slow_ratio_7d_rolling": "slow_vessel_ratio",
        "speed_7d_rolling":      "avg_speed_in_zone",
    }

    for new_col, src_col in rolling_map.items():
        if src_col in daily.columns:
            daily[new_col] = (
                daily.groupby("nearest_port", sort=False)[src_col]
                .transform(rolling_mean)
                .round(4)
            )
            log.info(f"  ✓ {new_col}")

    return daily


# ---------------------------------------------------------------------------
# Composite congestion score
# ---------------------------------------------------------------------------

def add_congestion_score(daily: pd.DataFrame) -> pd.DataFrame:
    """Weighted composite port_congestion_score in [0, 1]."""
    log.info("Computing port_congestion_score ...")

    idle_norm = (daily["avg_idle_duration"] / IDLE_DURATION_CAP).clip(0, 1)

    daily["port_congestion_score"] = (
        CONGESTION_WEIGHTS["slow_vessel_ratio"]  * daily["slow_vessel_ratio"].fillna(0)
        + CONGESTION_WEIGHTS["anchored_ratio"]   * daily["anchored_ratio"].fillna(0)
        + CONGESTION_WEIGHTS["idle_duration_norm"] * idle_norm.fillna(0)
    ).round(4)

    score = daily["port_congestion_score"]
    log.info(f"  Score range  : {score.min():.4f} → {score.max():.4f}")
    log.info(f"  Score mean   : {score.mean():.4f}")
    log.info(f"  Score median : {score.median():.4f}")

    return daily


# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------

def save_features(df: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    log.info(f"Saving {len(df):,} rows → {output_path}")
    df.to_parquet(output_path, index=False, engine="pyarrow", compression="snappy")
    size_mb = output_path.stat().st_size / 1e6
    log.info(f"  ✓ Saved — {size_mb:.2f} MB")


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

def print_summary(daily: pd.DataFrame) -> None:
    print("\n" + "=" * 60)
    print("  PORT FEATURES — ENGINEERING SUMMARY")
    print("=" * 60)
    print(f"  Total rows     : {len(daily):,}")
    print(f"  Ports covered  : {daily['nearest_port'].nunique()}")
    print(f"  Date range     : {daily['date'].min().date()} → {daily['date'].max().date()}")

    print("\n  Congestion score by port (mean):")
    score_by_port = (
        daily.groupby("nearest_port")["port_congestion_score"]
        .mean()
        .sort_values(ascending=False)
    )
    for port, score in score_by_port.items():
        bar = "█" * int(score * 30)
        print(f"    {port:<25} {score:.4f}  {bar}")

    print("\n  Feature null rates:")
    key_cols = [
        "vessels_in_zone", "slow_vessel_ratio", "anchored_ratio",
        "avg_speed_in_zone", "avg_idle_duration",
        "vessels_7d_rolling", "slow_ratio_7d_rolling",
        "port_congestion_score",
    ]
    for col in key_cols:
        if col in daily.columns:
            null_pct = daily[col].isnull().mean() * 100
            flag     = "  ⚠" if null_pct > 10 else "  ✓"
            print(f"  {flag}  {col:<30} {null_pct:.1f}% null")
    print("=" * 60)


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    config = load_config()

    vessel_path   = PROJECT_ROOT / config["paths"]["processed"] / "vessel_features.parquet"
    output_path   = PROJECT_ROOT / config["paths"]["processed"] / "port_features.parquet"
    ports         = config["ports"]
    radius_km     = config["anchorage"]["radius_km"]

    log.info("=" * 60)
    log.info("Phase 2 — Port Feature Engineering (Streaming)")
    log.info(f"  Anchorage radius : {radius_km} km")
    log.info(f"  Ports            : {[p['name'] for p in ports]}")
    log.info("=" * 60)

    if not vessel_path.exists():
        raise FileNotFoundError(
            f"vessel_features.parquet not found at {vessel_path}.\n"
            "Run vessel_features.py first."
        )

    # Step 1: Stream through parquet, aggregate each row group
    combined = stream_aggregate_all_batches(vessel_path, ports, radius_km)

    # Step 2: Reduce batch sums to correct daily totals
    daily = reduce_to_daily(combined)
    del combined

    # Step 3: Rolling features
    daily = add_rolling_port_features(daily)

    # Step 4: Congestion score
    daily = add_congestion_score(daily)

    # Step 5: Summary
    print_summary(daily)

    # Step 6: Save
    save_features(daily, output_path)

    log.info("Phase 2 Step 2 complete — port_features.parquet ready.")
    log.info("Next: run src/features/external_features.py")


if __name__ == "__main__":
    main()


  PORT FEATURES — ENGINEERING SUMMARY
  Total rows     : 450
  Ports covered  : 5
  Date range     : 2023-01-01 → 2023-03-31

  Congestion score by port (mean):
    Seattle                   0.9395  ████████████████████████████
    Los Angeles               0.9016  ███████████████████████████
    Savannah                  0.8852  ██████████████████████████
    Houston                   0.8523  █████████████████████████
    New York                  0.7799  ███████████████████████

  Feature null rates:
    ✓  vessels_in_zone                0.0% null
    ✓  slow_vessel_ratio              1.1% null
    ✓  anchored_ratio                 1.1% null
    ✓  avg_speed_in_zone              1.1% null
    ✓  avg_idle_duration              1.1% null
    ✓  vessels_7d_rolling             2.2% null
    ✓  slow_ratio_7d_rolling          2.2% null
    ✓  port_congestion_score          0.0% null


---
## Cell 7 — Run `external_features.py`

**What it does:** Engineers weather severity scores, operational risk flags, and calendar features from raw weather CSVs, then merges with port features.

**Expected time: < 2 minutes** — weather data is already small daily aggregations.

In [ ]:
print('=' * 60)
print('Running external_features.py ...')
print('=' * 60)

import subprocess
result = subprocess.run(
    ['python', 'src/features/external_features.py'],
    cwd=PROJECT_ROOT,
    capture_output=False,
    text=True
)

print('=' * 60)
if result.returncode == 0:
    parquet = Path(PROJECT_ROOT) / 'data/processed/external_features.parquet'
    if parquet.exists():
        size_mb = parquet.stat().st_size / 1e6

        import pandas as pd
        df = pd.read_parquet(parquet)
        print(f'✓ external_features.parquet saved — {size_mb:.2f} MB')
        print(f'  Rows           : {len(df):,}')
        print(f'  Columns        : {df.shape[1]}')

        # Check weather merge quality
        if 'max_weather_severity_score' in df.columns:
            null_pct = df['max_weather_severity_score'].isnull().mean() * 100
            print(f'  Weather merge  : {100-null_pct:.0f}% rows have weather data')
            if null_pct > 20:
                print(f'  ⚠ {null_pct:.0f}% null — port name mismatch likely. Check config.yaml port names.')
    else:
        print('✗ external_features.parquet not found — check logs above')
else:
    print(f'✗ Script exited with code {result.returncode}')
    print('Fix the error above before running the next cell.')

Running external_features.py ...
✓ external_features.parquet saved — 0.08 MB
  Rows           : 450
  Columns        : 36
  Weather merge  : 100% rows have weather data


---
## Cell 8 — Run `build_master_dataset.py`

**What it does:** Joins all three feature tables, engineers cross-source features, computes the binary `congestion_label`, runs a full quality report, saves `master_dataset.parquet`.

**Expected time: 5–10 minutes**



In [ ]:
"""
build_master_dataset.py
-----------------------
Phase 2 — Master Dataset Assembly Pipeline

Joins the three processed feature tables into a single master dataset,
engineers cross-source features, runs a quality report, and saves the
result to data/processed/master_dataset.parquet.

All three input files are small daily aggregations (port × day level).
This script never touches vessel_features.parquet — that heavy work
was already done by port_features.py.

Input files
-----------
    data/processed/port_features.parquet      ← tiny (~155 rows)
    data/processed/external_features.parquet  ← tiny (~155 rows)

Output
------
    data/processed/master_dataset.parquet
        One row per port per day.
        Contains all engineered features + binary congestion_label.

Cross-source features engineered here
--------------------------------------
    weather_congestion_interaction : weather_severity × slow_vessel_ratio
    vessel_weather_risk            : (1 - speed_drop) × weather_severity
    vessels_7d_rolling_norm        : vessels_7d_rolling normalised per port
    congestion_pressure_index      : weighted composite 0-1 index
    congestion_label               : binary target for classifier

Usage (from project root):
    python src/pipeline/build_master_dataset.py
"""

import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Path setup
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config_loader import load_config

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
# threshold derived dynamically from data
# Set to None to auto-compute from score distribution
# Or set a float (e.g. 0.91) to override manually
CONGESTION_THRESHOLD = None        # auto = 60th percentile of port_congestion_score
CONGESTION_THRESHOLD_PERCENTILE = 60   # top 40% of days = congested

PRESSURE_INDEX_WEIGHTS = {
    "port_congestion_score":      0.50,
    "max_weather_severity_score": 0.30,
    "vessels_7d_rolling_norm":    0.20,
}

NULL_WARNING_THRESHOLD = 0.10


# ---------------------------------------------------------------------------
# 1. Load and validate inputs
# ---------------------------------------------------------------------------

def load_inputs(processed_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load port_features and external_features parquets.

    vessel_features.parquet is intentionally NOT loaded here — it contains
    242M rows and would OOM. All vessel-level aggregations needed for the
    master dataset were already computed by port_features.py and are
    present in port_features.parquet.

    Returns
    -------
    tuple of (port_df, external_df)
    """
    files = {
        "port_features":     processed_dir / "port_features.parquet",
        "external_features": processed_dir / "external_features.parquet",
    }

    log.info("Loading input files ...")
    dataframes = {}

    for name, path in files.items():
        if not path.exists():
            raise FileNotFoundError(
                f"Required file not found: {path}\n"
                f"Run the corresponding script first:\n"
                f"  port_features     → python src/features/port_features.py\n"
                f"  external_features → python src/features/external_features.py"
            )

        df = pd.read_parquet(path, engine="pyarrow")
        log.info(f"  ✓ {name:<25} {len(df):>6,} rows  ×  {df.shape[1]} columns")
        dataframes[name] = df

    return dataframes["port_features"], dataframes["external_features"]


# ---------------------------------------------------------------------------
# 2. Merge feature tables
# ---------------------------------------------------------------------------

def merge_features(
    port_df:     pd.DataFrame,
    external_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    LEFT JOIN port_features with external_features on (nearest_port, date).

    port_features is the base table — every port-day row is retained.
    external_features (weather + calendar) is joined onto it.

    Both tables are already at the same daily port level so no aggregation
    is needed — this is a simple keyed join on two small DataFrames.

    Parameters
    ----------
    port_df : pd.DataFrame
        Daily port-level congestion features.
    external_df : pd.DataFrame
        Daily weather + calendar features.

    Returns
    -------
    pd.DataFrame
        Merged master DataFrame sorted by port and date.
    """
    log.info("Merging port_features + external_features ...")

    # Normalise date types
    port_df["date"]     = pd.to_datetime(port_df["date"]).dt.normalize()
    external_df["date"] = pd.to_datetime(external_df["date"]).dt.normalize()

    # Normalise port name column in external_df
    # external_features.py may have saved it as 'port' or 'nearest_port'
    # depending on whether port_features existed at merge time
    if "port" in external_df.columns and "nearest_port" not in external_df.columns:
        external_df = external_df.rename(columns={"port": "nearest_port"})

    if "nearest_port" not in external_df.columns:
        raise ValueError(
            "external_features.parquet has neither 'port' nor 'nearest_port' column.\n"
            f"Available columns: {list(external_df.columns)}\n"
            "Re-run external_features.py to regenerate."
        )

    # Drop columns that exist in both tables to avoid _x/_y suffixes
    # Keep port_df version of any overlap (it's the primary source)
    port_cols     = set(port_df.columns) - {"nearest_port", "date"}
    external_cols = set(external_df.columns) - {"nearest_port", "date"}
    overlap       = port_cols & external_cols

    if overlap:
        log.info(f"  Dropping {len(overlap)} overlapping columns from external: {sorted(overlap)}")
        external_df = external_df.drop(columns=list(overlap))

    master = port_df.merge(
        external_df,
        on=["nearest_port", "date"],
        how="left",
    )

    # Check merge quality
    if "max_weather_severity_score" in master.columns:
        weather_null_pct = master["max_weather_severity_score"].isnull().mean() * 100
        log.info(f"  Rows with weather data : {100 - weather_null_pct:.0f}%")
        if weather_null_pct > 20:
            log.warning(
                f"  ⚠ {weather_null_pct:.0f}% of rows missing weather data.\n"
                "    Port names in config.yaml may not match weather CSV filenames.\n"
                "    Check that fetch_weather.py used the same port names."
            )

    master.sort_values(["nearest_port", "date"], inplace=True)
    master.reset_index(drop=True, inplace=True)

    log.info(f"  Merged shape : {len(master):,} rows × {master.shape[1]} columns")

    return master


# ---------------------------------------------------------------------------
# 3. Cross-source feature engineering
# ---------------------------------------------------------------------------

def add_cross_source_features(master: pd.DataFrame) -> pd.DataFrame:
    """
    Engineer features that require columns from both port_features and
    external_features. These cannot be computed inside either individual
    feature script.

    Features added
    --------------
    weather_congestion_interaction
        max_weather_severity_score × slow_vessel_ratio
        Days with simultaneous bad weather and vessel congestion.

    vessels_7d_rolling_norm
        vessels_7d_rolling min-max normalised within each port.
        Puts different-sized ports on the same 0-1 traffic scale.

    congestion_pressure_index
        Unified 0-1 composite across port congestion, weather severity,
        and normalised vessel traffic volume.

    congestion_label
        Binary classification target for Phase 3.
        1 if port_congestion_score > CONGESTION_THRESHOLD else 0.
    """
    log.info("Engineering cross-source features ...")

    # ── Weather × congestion interaction ─────────────────────────────────────
    weather_col    = "max_weather_severity_score"
    slow_ratio_col = "slow_vessel_ratio"

    if weather_col in master.columns and slow_ratio_col in master.columns:
        master["weather_congestion_interaction"] = (
            master[weather_col].fillna(0) * master[slow_ratio_col].fillna(0)
        ).round(4)
        log.info("  ✓ weather_congestion_interaction")
    else:
        master["weather_congestion_interaction"] = np.nan
        missing = [c for c in [weather_col, slow_ratio_col] if c not in master.columns]
        log.warning(f"  weather_congestion_interaction → NaN (missing: {missing})")

    # ── Normalise vessels_7d_rolling within each port ─────────────────────────
    rolling_col = "vessels_7d_rolling"

    if rolling_col in master.columns:
        def minmax_norm(s: pd.Series) -> pd.Series:
            s_min, s_max = s.min(), s.max()
            if s_max == s_min:
                return pd.Series(0.5, index=s.index)
            return (s - s_min) / (s_max - s_min)

        master["vessels_7d_rolling_norm"] = (
            master.groupby("nearest_port")[rolling_col]
            .transform(minmax_norm)
            .round(4)
        )
        log.info("  ✓ vessels_7d_rolling_norm")
    else:
        master["vessels_7d_rolling_norm"] = np.nan
        log.warning(f"  vessels_7d_rolling_norm → NaN ('{rolling_col}' not found)")

    # ── Unified congestion pressure index ────────────────────────────────────
    components = {
        "port_congestion_score":      PRESSURE_INDEX_WEIGHTS["port_congestion_score"],
        weather_col:                  PRESSURE_INDEX_WEIGHTS["max_weather_severity_score"],
        "vessels_7d_rolling_norm":    PRESSURE_INDEX_WEIGHTS["vessels_7d_rolling_norm"],
    }

    available   = {col: w for col, w in components.items() if col in master.columns}
    missing     = [col for col in components if col not in master.columns]

    if missing:
        log.warning(f"  Pressure index: missing components {missing} — renormalising weights")

    if available:
        total_weight = sum(available.values())
        master["congestion_pressure_index"] = sum(
            master[col].fillna(0) * (w / total_weight)
            for col, w in available.items()
        ).clip(0, 1).round(4)
        log.info(f"  ✓ congestion_pressure_index  (components: {list(available.keys())})")
    else:
        master["congestion_pressure_index"] = np.nan
        log.warning("  congestion_pressure_index → NaN (no components available)")

    # ── Port-relative score normalisation ────────────────────────────────────
    # Normalise port_congestion_score within each port using min-max scaling.
    # This converts "absolute 0.91" into "how bad is today vs this port's range"
    # A score of 1.0 = worst day observed, 0.0 = best day observed.
    if "port_congestion_score" in master.columns:
        def minmax_port(s: pd.Series) -> pd.Series:
            s_min, s_max = s.min(), s.max()
            if s_max == s_min:
                # All days identical — no variance, assign 0.5 to everything
                log.warning(
                    "port_congestion_score has zero variance within a port. "
                    "All days will be labelled identically. "
                    "Consider downloading more days of AIS data for more variance."
                )
                return pd.Series(0.5, index=s.index)
            return (s - s_min) / (s_max - s_min)

        master["port_congestion_score_norm"] = (
            master.groupby("nearest_port")["port_congestion_score"]
            .transform(minmax_port)
            .round(4)
        )
        log.info("  ✓ port_congestion_score_norm  (min-max per port)")
    else:
        master["port_congestion_score_norm"] = np.nan

    # ── Dynamic congestion label ──────────────────────────────────────────────
    # Use normalised score for labelling so the threshold is meaningful
    # regardless of what the absolute score range is.
    score_col = "port_congestion_score_norm"

    if score_col in master.columns and master[score_col].notna().any():
        if CONGESTION_THRESHOLD is None:
            # Auto-compute threshold from the actual score distribution
            threshold = master[score_col].quantile(
                CONGESTION_THRESHOLD_PERCENTILE / 100
            )
            log.info(
                f"  Auto threshold : {CONGESTION_THRESHOLD_PERCENTILE}th percentile "
                f"of port_congestion_score_norm = {threshold:.4f}"
            )
        else:
            threshold = CONGESTION_THRESHOLD
            log.info(f"  Manual threshold : {threshold}")

        master["congestion_label"] = (
            master[score_col] > threshold
        ).astype(np.int8)

        counts   = master["congestion_label"].value_counts()
        n_pos    = counts.get(1, 0)
        n_neg    = counts.get(0, 0)
        n_total  = len(master)
        minority = min(n_pos, n_neg) / n_total * 100

        log.info(f"  ✓ congestion_label  (threshold={threshold:.4f} on normalised score)")
        log.info(f"     Label=1 (congested)     : {n_pos:>4}  ({n_pos/n_total*100:.1f}%)")
        log.info(f"     Label=0 (not congested) : {n_neg:>4}  ({n_neg/n_total*100:.1f}%)")

        if minority < 10:
            log.warning(
                f"  ⚠ Severe class imbalance — minority={minority:.1f}%.\n"
                f"    Score has very low variance within this port.\n"
                f"    Download more days of data (60-90 days) for better separation.\n"
                f"    In Phase 3 use scale_pos_weight in XGBoost."
            )
        elif minority < 25:
            log.warning(
                f"  ⚠ Moderate imbalance — minority={minority:.1f}%.\n"
                f"    Monitor precision-recall carefully in Phase 3."
            )
        else:
            log.info(f"  ✓ Label balance healthy — minority class = {minority:.1f}%")
    else:
        master["congestion_label"] = np.nan
        log.warning("  congestion_label → NaN (normalised score not available)")

    return master


# ---------------------------------------------------------------------------
# 4. Quality report
# ---------------------------------------------------------------------------

def run_quality_report(master: pd.DataFrame) -> None:
    """Comprehensive quality audit printed to stdout before saving."""

    print("\n" + "=" * 65)
    print("  MASTER DATASET — QUALITY REPORT")
    print("=" * 65)

    # Shape
    print(f"\n  Shape         : {len(master):,} rows  ×  {master.shape[1]} columns")
    print(f"  Ports         : {master['nearest_port'].nunique()}")
    print(f"  Date range    : {master['date'].min().date()} → {master['date'].max().date()}")

    # Null audit
    null_rates = master.isnull().mean().sort_values(ascending=False)
    high_null  = null_rates[null_rates > NULL_WARNING_THRESHOLD]

    print(f"\n  Null rate audit (threshold: >{NULL_WARNING_THRESHOLD*100:.0f}%):")
    if high_null.empty:
        print(f"    ✓ No columns exceed {NULL_WARNING_THRESHOLD*100:.0f}% null.")
    else:
        for col, rate in high_null.items():
            print(f"    ⚠  {col:<45} {rate*100:.1f}% null")

    # Key feature distributions
    key_cols = [
        "port_congestion_score", "slow_vessel_ratio",
        "avg_speed_in_zone", "vessels_in_zone",
        "max_weather_severity_score",
        "weather_congestion_interaction",
        "congestion_pressure_index",
    ]
    available = [c for c in key_cols if c in master.columns]

    if available:
        print(f"\n  Key feature distributions:")
        print(f"  {'Column':<40} {'Mean':>8} {'Median':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
        print(f"  {'─'*80}")
        for col in available:
            s = master[col].dropna()
            if len(s) > 0:
                print(
                    f"  {col:<40} "
                    f"{s.mean():>8.4f} {s.median():>8.4f} "
                    f"{s.std():>8.4f} {s.min():>8.4f} {s.max():>8.4f}"
                )

    # Label balance
    if "congestion_label" in master.columns:
        counts  = master["congestion_label"].value_counts().sort_index()
        n_total = len(master)
        threshold_used = CONGESTION_THRESHOLD if CONGESTION_THRESHOLD else f"{CONGESTION_THRESHOLD_PERCENTILE}th percentile"

        print(f"\n  Congestion label balance (threshold={threshold_used} on normalised score):")
        for label, count in counts.items():
            status = "CONGESTED    " if label == 1 else "NOT CONGESTED"
            pct    = count / n_total * 100
            bar    = "█" * int(pct / 2)
            print(f"    Label={label}  {status}  {count:>4}  ({pct:.1f}%)  {bar}")

    # Per-port coverage
    print(f"\n  Rows per port:")
    for port, count in master.groupby("nearest_port").size().sort_values(ascending=False).items():
        print(f"    {port:<25} {count:>4} days")

    # Date continuity
    print(f"\n  Date continuity:")
    all_dates = pd.date_range(master["date"].min(), master["date"].max(), freq="D")
    for port in master["nearest_port"].unique():
        port_dates   = set(master[master["nearest_port"] == port]["date"].dt.normalize())
        missing_days = len(set(all_dates.normalize()) - port_dates)
        icon         = "✓" if missing_days == 0 else f"⚠ {missing_days} missing days"
        print(f"    {port:<25} {icon}")

    print("\n" + "=" * 65)


# ---------------------------------------------------------------------------
# 5. Save
# ---------------------------------------------------------------------------

def save_master(master: pd.DataFrame, output_path: Path) -> None:
    """Save master dataset to Parquet + companion CSV sample."""
    output_path.parent.mkdir(parents=True, exist_ok=True)

    log.info(f"Saving master dataset → {output_path}")
    master.to_parquet(output_path, index=False, engine="pyarrow", compression="snappy")
    size_mb = output_path.stat().st_size / 1e6
    log.info(f"  ✓ Parquet saved — {size_mb:.2f} MB")

    sample_path = output_path.parent / "master_dataset_sample.csv"
    master.head(500).to_csv(sample_path, index=False)
    log.info(f"  ✓ Sample CSV (500 rows) → {sample_path.name}")

    # Final column manifest
    log.info(f"\n  Final columns ({len(master.columns)}):")
    for col in master.columns:
        null_rate = master[col].isnull().mean()
        null_info = f"  ← {null_rate*100:.0f}% null" if null_rate > 0.05 else ""
        log.info(f"    {col:<45} {str(master[col].dtype):<15}{null_info}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    config = load_config()

    processed_dir = PROJECT_ROOT / config["paths"]["processed"]
    output_path   = processed_dir / "master_dataset.parquet"

    log.info("=" * 65)
    log.info("Phase 2 — Master Dataset Assembly")
    log.info(f"  Congestion label threshold : {CONGESTION_THRESHOLD}")
    log.info(f"  Note: vessel_features.parquet is NOT loaded here.")
    log.info(f"        All vessel aggregations already in port_features.parquet.")
    log.info("=" * 65)

    # Step 1: Load the two small processed parquets
    port_df, external_df = load_inputs(processed_dir)

    # Step 2: Merge them
    master = merge_features(port_df, external_df)

    # Step 3: Cross-source features + congestion label
    master = add_cross_source_features(master)

    # Step 4: Quality report
    run_quality_report(master)

    # Step 5: Save
    save_master(master, output_path)

    log.info("\nPhase 2 complete.")
    log.info(f"  Master dataset : {output_path}")
    log.info(f"  Rows           : {len(master):,}")
    log.info(f"  Columns        : {master.shape[1]}")
    log.info(f"  Ready for Phase 3 model training.")


if __name__ == "__main__":
    main()


  MASTER DATASET — QUALITY REPORT

  Shape         : 450 rows  ×  41 columns
  Ports         : 5
  Date range    : 2023-01-01 → 2023-03-31

  Null rate audit (threshold: >10%):
    ✓ No columns exceed 10% null.

  Key feature distributions:
  Column                                       Mean   Median      Std      Min      Max
  ────────────────────────────────────────────────────────────────────────────────
  port_congestion_score                      0.8717   0.8938   0.1084   0.0000   0.9632
  slow_vessel_ratio                          0.8682   0.8846   0.0672   0.7084   0.9604
  avg_speed_in_zone                          1.3144   1.0570   0.7818   0.4310   3.3280
  vessels_in_zone                          390.1689 347.5000 209.0008   0.0000 794.0000
  max_weather_severity_score                 0.2152   0.1590   0.1326   0.0415   0.5940
  weather_congestion_interaction             0.1844   0.1369   0.1183   0.0000   0.5408
  congestion_pressure_index                  0.6158   0.625

---
## Cell 9 — Verify All Output Files

Confirms all five processed files exist with sensible sizes before downloading.

In [ ]:
import pandas as pd
from pathlib import Path

processed_dir = Path(PROJECT_ROOT) / 'data/processed'

expected_files = [
    'vessel_features.parquet',
    'port_features.parquet',
    'external_features.parquet',
    'master_dataset.parquet',
    'master_dataset_sample.csv',
]

print('Output file verification')
print('─' * 60)

all_present = True
for fname in expected_files:
    fpath = processed_dir / fname
    if fpath.exists():
        size_mb = fpath.stat().st_size / 1e6
        print(f'  ✓  {fname:<45} {size_mb:>8.2f} MB')
    else:
        print(f'  ✗  {fname:<45}  MISSING')
        all_present = False

print('─' * 60)

if all_present:
    print('\n✓ All output files present.')
    print('  These files are already saved to your Google Drive.')
    print('  Proceed to Cell 10 to download them to your laptop.')
else:
    print('\n✗ Some files are missing. Check the pipeline cells above for errors.')

# Quick master dataset preview
master_path = processed_dir / 'master_dataset.parquet'
if master_path.exists():
    master = pd.read_parquet(master_path)
    print(f'\nMaster dataset preview (first 3 rows):')
    key_cols = [
        'nearest_port', 'date', 'vessels_in_zone',
        'slow_vessel_ratio', 'port_congestion_score',
        'max_weather_severity_score', 'congestion_label'
    ]
    available_cols = [c for c in key_cols if c in master.columns]
    print(master[available_cols].head(3).to_string(index=False))

Output file verification
────────────────────────────────────────────────────────────
  ✓  vessel_features.parquet                        9742.76 MB
  ✓  port_features.parquet                             0.05 MB
  ✓  external_features.parquet                         0.08 MB
  ✓  master_dataset.parquet                            0.10 MB
  ✓  master_dataset_sample.csv                         0.10 MB
────────────────────────────────────────────────────────────

✓ All output files present.
  These files are already saved to your Google Drive.
  Proceed to Cell 10 to download them to your laptop.

Master dataset preview (first 3 rows):
nearest_port       date  vessels_in_zone  slow_vessel_ratio  port_congestion_score  max_weather_severity_score  congestion_label
     Houston 2023-01-01              455             0.9248                 0.9301                      0.1300                 1
     Houston 2023-01-02              437             0.9352                 0.9393                     

---
## Cell 10 — Download Processed Files to Laptop

This triggers browser download prompts for each processed file.

**Note:** `vessel_features.parquet` may be large (1–2 GB). The others are tiny.

**Save each file to:** `port-intelligence/data/processed/` on your laptop.

**Alternative:** If you prefer, just download from Google Drive directly in your browser — the files are already saved there. Navigate to `port-intelligence/data/processed/`, select all, right-click → Download.

In [ ]:
# Cell 10 — Download Processed Files to Laptop
#
# Zips all required processed files into one archive and triggers
# a single browser download. Much more reliable than one popup per file.
#
# What gets downloaded:
#   port_intelligence_processed.zip
#     ├── master_dataset.parquet        (primary input)
#     ├── master_dataset_sample.csv     (quick inspection)
#     ├── port_features.parquet         (port aggregations)
#     └── external_features.parquet     (weather + calendar features)
#
# vessel_features.parquet is intentionally excluded (3.3 GB — not needed
# locally).
# Download it manually from Drive only if you need to re-run the pipeline.

import zipfile
import os
import shutil
from pathlib import Path
from google.colab import files

processed_dir = Path(PROJECT_ROOT) / 'data/processed'
zip_path      = Path(PROJECT_ROOT) / 'port_intelligence_processed.zip'

# Files to include in the zip — order doesn't matter
DOWNLOAD_FILES = [
    'master_dataset.parquet',
    'master_dataset_sample.csv',
    'port_features.parquet',
    'external_features.parquet',
]

# ── Pre-flight check ──────────────────────────────────────────────────────────
print('Pre-flight check ...')
print('─' * 55)

all_found  = True
total_size = 0.0

for fname in DOWNLOAD_FILES:
    fpath = processed_dir / fname
    if fpath.exists():
        size_mb     = fpath.stat().st_size / 1e6
        total_size += size_mb
        print(f'  ✓  {fname:<45} {size_mb:>7.2f} MB')
    else:
        print(f'  ✗  {fname:<45} MISSING')
        all_found = False

print('─' * 55)
print(f'  Total uncompressed : {total_size:.2f} MB')

if not all_found:
    print('\n✗ Some files are missing.')
    print('  Run the pipeline cells (5–8) before downloading.')
    raise SystemExit('Aborting download — missing files.')

# ── Build zip archive ─────────────────────────────────────────────────────────
print(f'\nBuilding zip archive ...')

# Remove any previous zip from a prior run
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for fname in DOWNLOAD_FILES:
        fpath = processed_dir / fname
        # Store files flat inside zip (no subdirectory path)
        zf.write(fpath, arcname=fname)
        print(f'  + {fname}')

zip_size_mb = zip_path.stat().st_size / 1e6
print(f'\n✓ Archive created : {zip_path.name}  ({zip_size_mb:.2f} MB compressed)')

# ── Trigger single browser download ──────────────────────────────────────────
print(f'\nStarting download ...')
print(f'Save the file anywhere on your laptop — we will move it next.')
files.download(str(zip_path))

# ── Instructions printed after download starts ────────────────────────────────
print("""
  AFTER DOWNLOAD COMPLETES —

  1. Locate .zip in your Downloads folder

  2. Extract it — you will get these 4 files:
       master_dataset.parquet
       master_dataset_sample.csv
       port_features.parquet
       external_features.parquet

  3. Move all 4 files into:
       PROJECT_ROOT/data/processed/

  5. Verify your laptop folder looks like this:
       data/processed/
       ├── master_dataset.parquet
       ├── master_dataset_sample.csv
       ├── port_features.parquet
       └── external_features.parquet

  6. You do NOT need vessel_features.parquet locally.
     Phase 3 only reads master_dataset.parquet.


""")

# ── Cleanup zip from Drive (optional — saves Drive storage) ──────────────────
cleanup = True   # set to False if you want to keep the zip on Drive

if cleanup and zip_path.exists():
    zip_path.unlink()
    print(f'  Zip deleted from Drive (files still safe in data/processed/).')

---
## Troubleshooting Reference

| Error | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: src.config_loader` | PROJECT_ROOT not in sys.path | Re-run Cell 2 before running pipeline cells |
| `FileNotFoundError: config.yaml` | Wrong project root path | Check FOLDER_NAME in Cell 2 matches your Drive folder exactly |
| `No CSV files found in data/raw/ais/` | AIS folder empty or wrong path | Re-run Cell 4 to download files |
| `No vessel pings matched any port zone` | radius_km too small or wrong region | Confirm config.yaml has `radius_km: 25` and Los Angeles is in ports list |
| Cell 5 crashes / Colab disconnects | Out of RAM on free tier | Runtime → Change runtime type → High-RAM, or upgrade to Colab Pro |
| `100% congestion label` | radius_km too large (capturing open ocean) | Confirm config.yaml has `radius_km: 25` |
| Weather merge shows high nulls | Port name mismatch | Port names in config.yaml must exactly match what fetch_weather.py saved |